# 20 · Per-class reject thresholds for unfamiliar-fault detection (Stage R9)

**Question.** With the same total number of known recordings rejected, do per-class thresholds catch more unfamiliar
faults than today's one shared threshold?

Exploratory, development data only, **no training and no forward pass**: the saved recording scores of the 81
leave-one-class-out (LOCO) CNN-6 runs (9 withheld classes × 3 outer folds × seeds 42, 43, 44). Reserved tests stay sealed.

## Background

**Today.** The confidence gate has **one** MSP threshold τ_p and **one** Mahalanobis threshold τ_d per LOCO run, shared by
every class, each set at the 97.5 % order statistic of all the run's calibration recordings pooled.

**Why per class.** Known classes differ in spread. Imbalance covers 6–35 g and the misalignments cover many speeds, so
their scores spread wide; bearing faults are tight. One shared threshold can reject too many known recordings of the wide
classes and let through unknowns that land just outside the tight ones. **Per-class thresholds** give each *predicted*
class its own τ_p and τ_d, set from that class's calibration recordings only (class-conditional, or Mondrian, calibration).
The figure below shows one run's calibration distances per class with the shared τ_d and each class's own τ_d.

**Three known limits, stated before any result.**
1. **Small samples:** calibration has only a few recordings per class (printed below: about 7–10 for most fault classes,
   12–18 for vertical misalignment, 15–17 for imbalance, 2–3 for normal). A high quantile of a few values is unstable.
2. **R8 finding:** withheld vertical misalignment (AUROC below 0.5) and imbalance (about 0.59) lie *inside* known clusters
   under every distance tried. No threshold, shared or per class, can separate them.
3. **Q3 only:** this stage changes the Q3 gate only. The healthy check (Q1) has a single class, so it is out of scope.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> loads the saved recording scores of seed 42 and draws, for one run (withheld imbalance, fold 0), every calibration recording's Mahalanobis distance as a dot in its class's strip, with today's shared τ_d (dashed) and each class's own τ_d at the 0.975 quantile (short solid line). Saved to <code>results/research/figures/r9_thresholds_one_run.png</code>.</div>

In [ ]:
# Background figure: one run's calibration Mahalanobis per class, the shared τ_d and each class's own τ_d (q = 0.975).
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ml.paths import SYSTEM_CODE_DIR, results_dir
from ml.loco_eval import decide, recordings
from ml.classwise_eval import MIN_CAL, SECONDARY_Q, quantiles

FIG = results_dir() / "research" / "figures"
FIG.mkdir(parents=True, exist_ok=True)
rec42 = recordings(42)
run = rec42[(rec42.withheld == "imbalance") & (rec42.outer_fold == 0)]
cal = run[run.role == "cal"]
tau_d_shared = float(decide(run).tau_d.iloc[0])
classes = sorted(cal.class10.unique())
fig, ax = plt.subplots(figsize=(9, 3.8))
for i, c in enumerate(classes):
    v = cal[cal.class10 == c].maha.to_numpy()
    ax.scatter(np.full(len(v), i) + np.linspace(-0.12, 0.12, len(v)), v, s=14, color="C0")
    if len(v) >= MIN_CAL:
        ax.plot([i - 0.3, i + 0.3], [quantiles(v, SECONDARY_Q)] * 2, color="C2", lw=2)
ax.axhline(tau_d_shared, color="k", ls="--", lw=1.2, label=f"shared τ_d (today) = {tau_d_shared:.1f}")
ax.plot([], [], color="C2", lw=2, label="per-class τ_d at q = 0.975 (≥ 3 recordings)")
ax.set_xticks(range(len(classes)), [c.replace("_", " ") for c in classes], rotation=30, ha="right")
ax.set_ylabel("Mahalanobis distance (calibration)"); ax.set_title("Seed 42, withheld imbalance, fold 0: calibration recordings")
ax.legend(fontsize=8); fig.tight_layout(); fig.savefig(FIG / "r9_thresholds_one_run.png", dpi=150); plt.show()

## The declaration, fixed before any code or computation
Quoted directly from CLAUDE.md §7 (Stage R9 and its addendum, committed before any code).

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> reads CLAUDE.md and prints the committed Stage R9 protocol, including addendum 14 (the matched-q tie rule).</div>

In [ ]:
# Quote the committed Stage R9 declaration from CLAUDE.md.
claude = (SYSTEM_CODE_DIR.parent / "CLAUDE.md").read_text()
print(claude[claude.index("**Stage R9 protocol"):claude.index("## 7-prev12.")].strip())

## Load
The saved recording scores of all 81 runs and the provenance of this run. The notebook refuses to run on a working tree
with uncommitted changes (the recorded git sha must not be "dirty").

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> records the provenance (git sha, split hash, config sha) and stops if the tree has uncommitted changes, then loads the saved recording scores of the three seeds and prints the recordings per role and the calibration recordings per class (minimum, median and maximum over the 27 runs).</div>

In [ ]:
# Provenance (must be a clean committed tree), the saved scores of all 81 runs, counts per role and per class.
from ml.config import config_sha256, load_config
from ml.paths import DEFAULT_CONFIG
from ml.splits import split_hash
from ml.utils import git_sha
from ml.classwise_eval import ARMS, ARM_NAMES, calibration_counts

prov = {"git_sha": git_sha(), "split_hash": split_hash(pd.read_csv(SYSTEM_CODE_DIR / "metadata.csv")),
        "config_sha256": config_sha256(load_config(DEFAULT_CONFIG))}
print(prov)
assert not prov["git_sha"].endswith("-dirty"), "commit or set aside every change first: the run needs a clean tree"
recs = {s: recordings(s) for s in (42, 43, 44)}
print({s: r.groupby("role").size().to_dict() for s, r in recs.items()})
cal_counts = calibration_counts(pd.concat(recs.values(), ignore_index=True))
cal_summary = cal_counts[cal_counts.seed == 42].groupby("class10").n_cal.agg(["min", "median", "max"])
cal_summary

## Reproduction check (before anything else)
Arm G recomputed with `ml.loco_eval.decide` must equal the saved variant-B decisions of every recording of every run (the
Stage 3D files, which hold `decision_B` for all three seeds), and for seed 42 also variant A against the Stage 3C file.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> recomputes today's thresholds and decisions for every run with <code>ml.loco_eval.decide</code> and compares rule B with the saved variant-B decisions (all seeds), and rule A with the saved Stage 3C decisions (seed 42). The assert stops the notebook on any mismatch.</div>

In [ ]:
# Reproduce G: recomputed rule-B decisions must equal the saved ones on every recording (and rule A for seed 42).
from ml.paths import results_dir as _rd
dec = {s: decide(r) for s, r in recs.items()}
key = ["withheld", "outer_fold", "rec_id"]
repro = {}
for s, d in dec.items():
    saved = pd.read_parquet(_rd() / "scores" / f"e3_b1s_veto_s{s}_loco_decisions.parquet")[key + ["decision_B"]]
    j = d[key + ["decision_B"]].merge(saved, on=key, suffixes=("", "_saved"), validate="1:1")
    repro[s] = {"rows": len(j), "mismatch_B": int((j.decision_B != j.decision_B_saved).sum())}
s3c = pd.read_parquet(_rd() / "scores" / "cnn6ae6_s42_loco_decisions.parquet")[key + ["decision_classifier_first"]]
j = dec[42][key + ["decision_A"]].merge(s3c, on=key, validate="1:1")
repro[42]["mismatch_A_vs_3C"] = int((j.decision_A != j.decision_classifier_first).sum())
print(repro)
assert all(v["rows"] == len(dec[s]) and v["mismatch_B"] == 0 for s, v in repro.items()) and repro[42]["mismatch_A_vs_3C"] == 0

## Thresholds of one run
The per-class, shrunk and global thresholds of the example run at q = 0.975, with each class's number of calibration
recordings and whether it falls back to the global thresholds (fewer than 3).

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> builds, for the example run, a table of each class's calibration count and its τ_p and τ_d under P (per class) and S (shrunk), next to the global values at the same level and today's shared thresholds.</div>

In [ ]:
# One run's thresholds at q = 0.975: per class (P), shrunk (S), global, and today's τ_p, τ_d.
from ml.classwise_eval import thresholds
classes = sorted(set(run.pred) | set(cal.class10))
tp, ts = thresholds(cal, SECONDARY_Q, "P", classes), thresholds(cal, SECONDARY_Q, "S", classes)
g0 = decide(run).iloc[0]
pd.DataFrame({c: {"n_cal": int((cal.class10 == c).sum()), "fallback": c in tp["fallback"],
                  "P_tau_p": tp["per_class"][c][0], "P_tau_d": tp["per_class"][c][1],
                  "S_tau_p": ts["per_class"][c][0], "S_tau_d": ts["per_class"][c][1]} for c in classes}).T.assign(
    global_tau_p=tp["global"][0], global_tau_d=tp["global"][1], today_tau_p=g0.tau_p, today_tau_d=g0.tau_d).round(4)

## Primary result: the matched rejection budget
For each run, P and S get one level q each, chosen so that they reject as many fold-k known recordings as today's
thresholds do (G). At that q: unknown recall on the withheld recordings, withheld recordings called healthy, and known
recordings called the wrong fault, pooled over the 27 runs of each seed.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> evaluates every run once (<code>ml.classwise_eval.evaluate_run</code>: G's saved-rule decisions, then P and S at the matched q and at q = 0.975), then pools the matched outcomes per seed and arm, and summarises the q chosen and how close the reached rejection count came to G's.</div>

In [ ]:
# Evaluate all 81 runs once; matched-budget outcomes per seed and arm; the q and count difference reached.
import time
from ml.classwise_eval import evaluate_run
from ml.rmd_eval import pooled
t0 = time.perf_counter()
rows = []
for s, d in dec.items():
    for (c, k), g in d.groupby(["withheld", "outer_fold"]):
        g = g.reset_index(drop=True)
        for r in evaluate_run(g, g.decision_B.to_numpy()):
            rows.append({"seed": s, "withheld": c, "outer_fold": k, **r})
per_run = pd.DataFrame(rows)
run_seconds = time.perf_counter() - t0
print(f"evaluated {per_run.groupby(['seed', 'withheld', 'outer_fold']).ngroups} runs in {run_seconds:.0f} s")
matched = pooled(per_run, "matched")
qsum = per_run[per_run.arm != "G"].groupby(["seed", "arm"]).agg(q_median=("q", "median"), q_min=("q", "min"), q_max=("q", "max"),
        count_diff_total=("diff", "sum"), runs_not_exact=("diff", lambda x: int((x != 0).sum())))
display(matched.round(4))
qsum.round(4)

## Per-class unknown recall
Unknown recall per withheld class at the matched budget (pooled over the class's 3 folds within a seed, then the mean over
seeds), G against P against S; classes where P or S differ from G by 5 points or more are marked.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> computes unknown recall per withheld class for every arm at the matched budget, the mean over seeds, marks differences of 5 points or more against G, and draws the three arms as grouped bars (<code>results/research/figures/r9_recall_per_class.png</code>).</div>

In [ ]:
# Per-class unknown recall (matched budget): per seed, the seed mean, P − G and S − G with ≥ 5-point differences marked.
from ml.classwise_eval import withheld_recall_by_class
cls = withheld_recall_by_class(per_run, "matched")[list(ARMS)]
cls_mean = cls.groupby(level="withheld").mean()
for a in ("P", "S"):
    cls_mean[f"{a}_minus_G"] = cls_mean[a] - cls_mean.G
    cls_mean[f"{a}_mark"] = np.where(cls_mean[f"{a}_minus_G"] >= 0.05, "better ≥ 5", np.where(cls_mean[f"{a}_minus_G"] <= -0.05, "worse ≥ 5", ""))
fig, ax = plt.subplots(figsize=(10, 3.8))
cls_mean[list(ARMS)].plot.bar(ax=ax, width=.8)
ax.set_ylabel("unknown recall (matched budget)"); ax.set_xlabel("withheld class"); ax.set_ylim(0, 1)
ax.set_title("Unknown recall per withheld class, mean over seeds"); ax.legend([ARM_NAMES[a] for a in ARMS], fontsize=8)
plt.xticks(rotation=30, ha="right"); fig.tight_layout(); fig.savefig(FIG / "r9_recall_per_class.png", dpi=150); plt.show()
display(cls.unstack("seed").round(3))
cls_mean.round(3)

## Secondary result: per-class thresholds as they would be deployed (q = 0.975, no matching)
The known-side rejections are no longer equal across arms; their rate is shown next to the other outcomes.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> pools the q = 0.975 outcomes per seed and arm and adds the known-side rejection rate (fold-k known recordings sent to further inspection); G's row is today's rule.</div>

In [ ]:
# Secondary: P and S at q = 0.975 with no matching, with the known-side rejection rate.
secondary = pooled(per_run, "secondary")
secondary["known_rejection_rate"] = secondary.known_rejected / secondary.known_n
secondary.round(4)

## Diagnostics (descriptive only)
Calibration recordings per class, how often a class fell back to the global thresholds, and how much the per-class τ_d
really differs across classes within a run.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> counts the class-runs that fell back to the global thresholds (fewer than 3 calibration recordings), and summarises, over the 81 runs, the spread of the per-class τ_d at q = 0.975 (smallest, largest and their ratio) against the global value.</div>

In [ ]:
# Diagnostics: fallbacks per class and seed, and the spread of the per-class τ_d within a run.
from ml.classwise_eval import tau_d_spread
fallbacks = cal_counts.assign(fallback=cal_counts.n_cal < MIN_CAL).groupby(["seed", "class10"]).fallback.sum().unstack("seed")
print("class-runs that fall back to the global thresholds (per seed):", cal_counts.assign(f=cal_counts.n_cal < MIN_CAL).groupby("seed").f.sum().to_dict())
spread = pd.DataFrame([{"seed": s, "withheld": c, "outer_fold": k, **tau_d_spread(g)}
                       for s, r in recs.items() for (c, k), g in r.groupby(["withheld", "outer_fold"])])
display(fallbacks[fallbacks.sum(axis=1) > 0])
spread[["tau_d_min", "tau_d_max", "tau_d_ratio_max_min", "tau_d_global"]].describe().round(2)

## Reading against the declared bar
Applied to P and S separately against G at the matched budget: **helps, confirmed** only if unknown recall is at least 5
points higher and no more withheld recordings are called healthy, on all three seeds; otherwise **worse** if the mean change
is negative; otherwise **does not help**. Both readings stand.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> applies the declared bar to P and to S (<code>ml.classwise_eval.reading</code>) and prints, per seed, the recall gain over G, the change in withheld recordings called healthy, and each arm's reading.</div>

In [ ]:
# The declared readings, applied mechanically to P and S.
from ml.classwise_eval import reading
readings = {a: reading(matched, a) for a in ("P", "S")}
for a, v in readings.items():
    for s, x in v["per_seed"].items():
        print(f"{a} seed {s}: unknown recall {a} − G = {x['recall_gain'] * 100:+.1f} points; withheld called healthy "
              f"{a} − G = {x['healthy_change']:+d}; bar met: {x['bar_met']}")
    print(f"{a}: mean recall change {v['mean_recall_gain'] * 100:+.1f} points  →  READING: {v['reading'].upper()}\n")

**In plain words.** _(written after the run; see the printed readings above)_

## Limitations
- Development data only; nothing here is an independent claim (reserved tests sealed; the final evaluation already ran).
- Few calibration recordings per class (7–10 for most fault classes, 2–3 for normal, which therefore always uses the global
  thresholds); high quantiles of so few values are unstable.
- The same LOCO partition, folds and saved models as Stages 3C and R4–R8.
- R8's finding stands: withheld vertical misalignment and imbalance sit inside known clusters, so no threshold can separate
  them; per-class thresholds can only redistribute the budget among the other classes.
- The shrink weight (0.5), the fallback minimum (3) and the quantile method are fixed, not tuned; a negative result says
  these per-class thresholds do not help here, not that no class-conditional rule could.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> writes the per-run table to <code>results/research/r9_per_run.csv</code> and every reported number, with provenance, the calibration counts, the diagnostics and both readings, to <code>results/research/r9_classwise_thresholds.json</code>.</div>

In [ ]:
# Write the outputs: per-run CSV and the JSON summary.
out = results_dir() / "research"
per_run.to_csv(out / "r9_per_run.csv", index=False)
summary = {"stage": "R9", "provenance": prov | {"run_seconds": round(run_seconds, 1)}, "arms": ARM_NAMES,
           "reproduction": repro, "calibration_counts": cal_counts.to_dict(orient="records"),
           "calibration_summary_seed42": cal_summary.reset_index().to_dict(orient="records"),
           "matched": matched.to_dict(orient="records"), "matched_q": qsum.reset_index().to_dict(orient="records"),
           "secondary": secondary.to_dict(orient="records"),
           "recall_per_class_matched": cls.reset_index().to_dict(orient="records"),
           "recall_per_class_seed_mean": cls_mean.reset_index().to_dict(orient="records"),
           "diagnostics": {"fallback_class_runs": fallbacks.fillna(0).astype(int).reset_index().to_dict(orient="records"),
                           "tau_d_spread": spread.to_dict(orient="records")},
           "readings": readings}
(out / "r9_classwise_thresholds.json").write_text(json.dumps(summary, indent=1, default=float))
print("written:", (out / "r9_per_run.csv").relative_to(SYSTEM_CODE_DIR), (out / "r9_classwise_thresholds.json").relative_to(SYSTEM_CODE_DIR))

## Outputs written
- `results/research/r9_classwise_thresholds.json`, `results/research/r9_per_run.csv`
- Figures: `results/research/figures/r9_thresholds_one_run.png`, `r9_recall_per_class.png`